In [4]:
import random, pickle, mygene, umap, sklearn, hnswlib, openai # use hnswlib for NN classification
import pandas as pd, numpy as np, scanpy as sc
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.cluster import MiniBatchKMeans
from xgboost import XGBClassifier
from scipy.stats import mode
# import sentence_transformers
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('retina') # for high resolution plots
sns.set_style("whitegrid")
plt.style.use('ggplot') # plt.style.use('seaborn-v0_8-dark-palette')
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams.update({"text.usetex": False, "font.family": "Helvetica"})

# Load sampled Aorta data

Available at https://drive.google.com/drive/folders/1LgFvJqWNq9BqHbuxB2tYf62kXs9KqL4t?usp=share_link. Downloaded from Li et al. "Single-Cell Transcriptome Analysis Reveals Dynamic Cell Populations and Differential Gene Expression Patterns in Control and Aneurysmal Human Aortic Tissue" (2020). (link: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE155468; https://github.com/LI-Yan-Ming/Circulation.-2020-142-1374-1388; https://www.ahajournals.org/doi/10.1161/CIRCULATIONAHA.120.046528)

In [ ]:
sampled_adata = sc.read_h5ad("./input_data/sample_aorta_data_updated.h5ad")
mapping_dict_phenotype = {'affected_1': 'Ascending only', 'affected_2': 'Ascending only', 'affected_3': 'Ascending w/ root', 'affected_4': 'Ascending only', 'affected_5': 'Ascending w/ root', 
        'affected_6': 'Ascending to descending', 'affected_7': 'Ascending w/ root', 'affected_8': 'Ascending w/ root', 'control_1': 'Control', 'control_2': 'Control', 'control_3': 'Control'}
sampled_adata.obs['phenotype'] = sampled_adata.obs.patient.map(mapping_dict_phenotype)
cell_type_list_phenotype = np.array(sampled_adata.obs.phenotype) # list of phenotypes
cell_type_list_celltype = np.array(sampled_adata.obs.celltype) # list of cell types
cell_type_list_patient = np.array(sampled_adata.obs.patient) # list of patients
gene_name_list = np.array(sampled_adata.var.index) # list of gene names
cell_data_x = sampled_adata.X # shape: (n_cells, n_genes)

### GenePT-s embeddings and GenePT-w embeddings
cell_embeddings_genept_s = [] # shape: (n_cells, 1536)
client = openai.OpenAI(api_key="")
for cell_data in cell_data_x:
    gene_indices = np.argsort(cell_data)[::-1] # sort by expression in descending order
    filtered_genes = gene_indices[~np.isin(gene_indices, np.where(cell_data == 0)[0])] # filter out genes with expression 0
    selected_genes = gene_name_list[filtered_genes][0:1000] # select top 1000 genes by expression
    cell_seqprompt = 'A cell with genes ranked by expression: ' + ' '.join(selected_genes) # format the cell prompt sequence for GPT-3.5
    cell_embedding = np.array(client.embeddings.create(input = [cell_seqprompt], model="text-embedding-ada-002").data[0].embedding) # get the embedding of the cell prompt sequence
    cell_embeddings_genept_s.append(cell_embedding)
cell_embeddings_genept_s = np.array(cell_embeddings_genept_s) # convert the list of embeddings to a numpy array
cell_embeddings_genept_w = [] # shape: (n_cells, 1536)
with open("./input_data/GenePT_embedding/GPT_3_5_gene_embeddings.pickle", "rb") as fp:
    gene_embeddings = pickle.load(fp) 
count_missing = sum([temp not in gene_embeddings for temp in gene_name_list])
print(f"Unable to match {count_missing} out of {len(gene_name_list)} genes in the GenePT-w embedding")
lookup_embed = np.zeros(shape=(len(gene_name_list), 1536)) # embedding dim from GPT-3.5 is 1536
for i, gene in enumerate(gene_name_list):
    lookup_embed[i,:] = gene_embeddings[gene].flatten() if gene in gene_embeddings else np.zeros(1536)
cell_embeddings_genept_w = np.dot(cell_data_x, lookup_embed)/len(gene_name_list) # GenePT-w embedding is the dot product of the cell expression matrix and the gene embeddings

In [ ]:
### UMAP visualizations by phenotype, cell type, and patient id
pca = sklearn.decomposition.PCA(n_components=50)
pca_result = pca.fit_transform(cell_embeddings_genept_s)
umap_result = umap.UMAP(n_components=2, min_dist=0.5, spread=1).fit_transform(pca_result) # Compute UMAP on the PCA-reduced data
plt.figure(figsize=(6,5)) 
for i, label_name in enumerate(np.unique(cell_type_list_phenotype)):
    plt.scatter(umap_result[cell_type_list_phenotype==label_name, 0], umap_result[cell_type_list_phenotype==label_name, 1], s= 2, label=label_name, color=sns.color_palette('husl')[i], alpha=0.2)
plt.xticks([]); plt.yticks([]); plt.legend(prop={'size': 10}, loc='lower left', ncol=2, markerscale=2.0)
plt.figure(figsize=(6,5)) 
for i, label_name in enumerate(np.unique(cell_type_list_celltype)):
    plt.scatter(umap_result[cell_type_list_celltype==label_name, 0], umap_result[cell_type_list_celltype==label_name, 1], s= 2, label=label_name, color=sns.color_palette('husl')[i], alpha=0.2)
plt.xticks([]); plt.yticks([]); plt.legend(prop={'size': 8}, loc='lower right', ncol=2, markerscale=2.0)
plt.figure(figsize=(6,5)) 
for i, label_name in enumerate(np.unique(cell_type_list_patient)):
    plt.scatter(umap_result[cell_type_list_patient==label_name, 0], umap_result[cell_type_list_patient==label_name, 1], s= 2, label=label_name, color=sns.color_palette('husl')[i], alpha=0.2)
plt.xticks([]); plt.yticks([]); plt.legend(prop={'size': 10}, loc='lower left', ncol=2, bbox_to_anchor=(1.1, 0.10), markerscale=2.0)

### Add quantitative data to measure the batch effect and the amount of biology encoded in the embeddings
# oh super high accuracy! interesting... !!!!!!!!
X_train, X_test, y_train, y_test = train_test_split(cell_embeddings_genept_s, sampled_adata.obs.phenotype, test_size=0.20, random_state=2023)
print(f"Training set size (X_train): {len(X_train)} \n Test set size (X_test): {len(X_test)}")
lr = LogisticRegression(max_iter=100); lr.fit(X_train, y_train); y_pred_lr = lr.predict(X_test)
print('accuracy', np.mean(y_test== y_pred_lr), 'precision', sklearn.metrics.precision_recall_fscore_support(y_test, y_pred_lr, average='weighted'))

# Correlate estimated clusters to patient-level effect
kmeans = MiniBatchKMeans(n_clusters=11, random_state=2023, batch_size=20); kmeans.fit(cell_embeddings_genept_s)
print('aRI', sklearn.metrics.adjusted_rand_score(kmeans.labels_, cell_type_list_patient),'aMI', sklearn.metrics.adjusted_mutual_info_score(kmeans.labels_, cell_type_list_patient))
# Correlate estimated clusters to cell types
kmeans = MiniBatchKMeans(n_clusters=11, random_state=2023, batch_size=20); kmeans.fit(cell_embeddings_genept_s[np.where(cell_type_list_celltype!='Unknown')[0]])
print('aRI', sklearn.metrics.adjusted_rand_score(kmeans.labels_, cell_type_list_celltype[np.where(cell_type_list_celltype!='Unknown')[0]]),'aMI', sklearn.metrics.adjusted_mutual_info_score(kmeans.labels_, cell_type_list_celltype[np.where(cell_type_list_celltype!='Unknown')[0]]))
# Correlate estimated clusters to phenotypes
kmeans = MiniBatchKMeans(n_clusters=4, random_state=2023, batch_size=20); kmeans.fit(cell_embeddings_genept_s)
print('aRI', sklearn.metrics.adjusted_rand_score(kmeans.labels_, cell_type_list_phenotype),'aMI', sklearn.metrics.adjusted_mutual_info_score(kmeans.labels_, cell_type_list_phenotype))
# print(pd.crosstab(labels, disease_label)) # print the confusion matrix

# Cell type annotation task

In this block, we demonstrate the performance of GenePT-s and GenePT-w embeddings on cell type annotation tasks in the Aorta dataset. 
We split the Aorta dataset with annotated cell types into 80%/20% train/test split and perform a 10-nearest-neighbor classifier. 
Code credit: the set up for the 10-NN classifier was ported from the scGPT authors at https://github.com/bowang-lab/scGPT/blob/main/tutorials/Tutorial_Reference_Mapping.ipynb

In [ ]:
def get_similar_vectors(query_vector, reference_vectors, k=10): # Get the indices of the k-nearest neighbors of a query vector in a reference set.
    distances = np.linalg.norm(reference_vectors - query_vector, axis=1) # shape: (n_reference_vectors,), L2 distance
    return np.argsort(distances)[:k], distances[np.argsort(distances)[:k]] # return the indices and distances of the k-nearest neighbors

def get_similar_vectors_by_hnswlib(query_vector, reference_vectors, k=10): # Get the indices of the k-nearest neighbors of a query vector in a reference set.
    p = hnswlib.Index(space = 'cosine', dim = reference_vectors.shape[1]) # possible options are l2, cosine or ip
    p.init_index(max_elements = reference_vectors.shape[0], ef_construction = 200, M = 16)
    p.add_items(reference_vectors, ids = np.arange(reference_vectors.shape[0])) # Element insertion (can be called several times):
    p.set_ef(50) # ef should always be > k
    index, distance = p.knn_query(query_vector, k = k) # Query dataset, k - number of closest elements (returns 2 numpy arrays)
    return index, distance

#### Cell type annotation with GenePT-w embeddings
x = cell_embeddings_genept_w[np.where(sampled_adata.obs.celltype!='Unknown')[0]]
y = sampled_adata.obs.celltype[np.where(sampled_adata.obs.celltype!='Unknown')[0]]
embed_train, embed_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=2023)
gt_list = []; pred_list = []; neighbors_list_gpt_v1 = []
for i in range(embed_test.shape[0]):
    gt = y_test[i]; idx, _= get_similar_vectors(embed_test[i][np.newaxis, ...], embed_train); pred = mode(y_train[idx], axis=0) # mode: most frequent element
    neighbors_list_gpt_v1.append(y_train[idx]); gt_list.append(gt); pred_list.append(pred[0][0]) # pred[0][0]: most frequent element
print('Accuracy for GenePT-w embedding: ', sklearn.metrics.accuracy_score(gt_list, pred_list), 'Precision, Recall, F1 (Marco weighted) for GenePT-w embedding: ', sklearn.metrics.precision_recall_fscore_support(gt_list, pred_list,average='macro'))

#### Cell type annotation with GenePT-s embeddings
x = cell_embeddings_genept_s[np.where(sampled_adata.obs.celltype!='Unknown')[0]]
y = sampled_adata.obs.celltype[np.where(sampled_adata.obs.celltype!='Unknown')[0]]
embed_train, embed_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=2023)
gt_list = []; pred_list = []; neighbors_list_gpt_v2 = []; 
for i in range(embed_test.shape[0]):
    gt = y_test[i]; idx, _= get_similar_vectors(embed_test[i][np.newaxis, ...], embed_train); pred = mode(y_train[idx], axis=0)
    neighbors_list_gpt_v2.append(y_train[idx]); gt_list.append(gt); pred_list.append(pred[0][0])
print('Accuracy for GenePT-s embedding: ', sklearn.metrics.accuracy_score(gt_list, pred_list), 'Precision, Recall, F1 (Marco weighted) for GenePT-w embedding: ', sklearn.metrics.precision_recall_fscore_support(gt_list, pred_list,average='macro'))